# Fabric Table Preview with PySpark

command to run the jupyter server for this notebook: 

`runjupyter (in any folder)`

equivalent to running this in /go-api:

```docker compose run --rm -p 8888:8888 serve jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root --notebook-dir=/home/ifrc```


In [14]:
import os
from pathlib import Path
from urllib.request import urlretrieve

POSTGRES_JDBC_VERSION = '42.7.4'
POSTGRES_JDBC_JAR = Path(f'/tmp/postgresql-{POSTGRES_JDBC_VERSION}.jar')
POSTGRES_JDBC_URL = (
    'https://repo1.maven.org/maven2/org/postgresql/postgresql/'
    f'{POSTGRES_JDBC_VERSION}/postgresql-{POSTGRES_JDBC_VERSION}.jar'
)

if not POSTGRES_JDBC_JAR.exists():
    urlretrieve(POSTGRES_JDBC_URL, POSTGRES_JDBC_JAR)

# Must be set before Spark JVM starts in this kernel
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--jars {POSTGRES_JDBC_JAR} pyspark-shell'

from pyspark.sql import SparkSession

In [15]:
# Stop the current Spark session so this cell can be rerun safely
active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

SPARK_MASTER = os.getenv("SPARK_MASTER", "local[*]")  # go to .env in go-api to configure, default to local 

spark = (
    SparkSession.builder
    .appName('postgres-stock-inventory')
    .master(SPARK_MASTER)
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark

In [16]:
# These are provided by docker compose env + .env when running notebook in `serve` container
PG_HOST = os.getenv('DJANGO_DB_HOST', 'db')
PG_PORT = os.getenv('DJANGO_DB_PORT', '5432')
PG_DB = os.environ['DJANGO_DB_NAME']
PG_USER = os.environ['DJANGO_DB_USER']
PG_PASSWORD = os.environ['DJANGO_DB_PASS']

jdbc_url = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'

# Read only table names in public schema to discover what you can load
table_list_query = """(
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
) t"""

tables_df = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_list_query)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Total public tables:', tables_df.count())
tables_df.show(5, truncate=False)

Total public tables: 351
+------------------------+
|table_name              |
+------------------------+
|api_action              |
|api_actionstaken        |
|api_actionstaken_actions|
|api_admin2              |
|api_admin2geoms         |
+------------------------+
only showing top 5 rows



In [17]:
# Example: load one simple Django table from go-api/api/models.py
table_name = 'api_CleanedFrameworkAgreement'

df_sample = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_name)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Table:', table_name)
df_sample.show(10, truncate=False)

Table: api_CleanedFrameworkAgreement
+----+------------+--------------+-------------------------------------+--------------------------------------+---------------+---------+--------------+----------------------------+------------------------+--------------+------------------------+---------+------------------+------------------------------+--------------------------+--------------------------+-----+-------------------+-------------------+
|id  |agreement_id|classification|default_agreement_line_effective_date|default_agreement_line_expiration_date|workflow_status|status   |price_per_unit|pa_line_procurement_category|vendor_name             |vendor_country|region_countries_covered|item_type|item_category     |item_service_short_description|created_at                |updated_at                |owner|vendor_valid_from  |vendor_valid_to    |
+----+------------+--------------+-------------------------------------+--------------------------------------+---------------+---------+------------

In [ ]:
# Load all required tables from Postgres as DataFrames
print("Loading tables from Postgres...")

dimwarehouse_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimwarehouse")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproduct_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproduct")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventorytransactionline_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventorytransactionline")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventorytransaction_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventorytransaction")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryowner_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryowner")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproductcategory_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproductcategory")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryitem_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryitem")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryitemstatus_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryitemstatus")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproductreceiptline_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproductreceiptline")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

print("✓ All tables loaded successfully")
print(f"  - dimwarehouse: {dimwarehouse_df.count()} rows")
print(f"  - dimproduct: {dimproduct_df.count()} rows")
print(f"  - diminventorytransactionline: {diminventorytransactionline_df.count()} rows")
print(f"  - diminventorytransaction: {diminventorytransaction_df.count()} rows")
print(f"  - diminventoryowner: {diminventoryowner_df.count()} rows")
print(f"  - dimproductcategory: {dimproductcategory_df.count()} rows")
print(f"  - diminventoryitem: {diminventoryitem_df.count()} rows")
print(f"  - diminventoryitemstatus: {diminventoryitemstatus_df.count()} rows")
print(f"  - dimproductreceiptline: {dimproductreceiptline_df.count()} rows")


Loading tables from Postgres...
✓ All tables loaded successfully
  - dimwarehouse: 1172 rows
  - dimproduct: 27640 rows
  - diminventorytransactionline: 24799 rows
  - diminventorytransaction: 21636 rows
  - diminventoryowner: 109 rows
  - dimproductcategory: 3273 rows


In [ ]:
# Register all DataFrames as SQL temp views
dimwarehouse_df.createOrReplaceTempView("dimwarehouse")
dimproduct_df.createOrReplaceTempView("dimproduct")
diminventorytransactionline_df.createOrReplaceTempView("diminventorytransactionline")
diminventorytransaction_df.createOrReplaceTempView("diminventorytransaction")
diminventoryowner_df.createOrReplaceTempView("diminventoryowner")
dimproductcategory_df.createOrReplaceTempView("dimproductcategory")
diminventoryitem_df.createOrReplaceTempView("diminventoryitem")
diminventoryitemstatus_df.createOrReplaceTempView("diminventoryitemstatus")
dimproductreceiptline_df.createOrReplaceTempView("dimproductreceiptline")

print("✓ All tables registered as SQL views")


✓ All tables registered as SQL views


In [25]:
dimproduct_df.show(5)

+--------------+--------------------+----+---------------+----------------+--------------------+
|            id|                name|type|unit_of_measure|product_category|    project_category|
+--------------+--------------------+----+---------------+----------------+--------------------+
|   AAUDLOUDLSM|LOUDSPEAKER Foste...|Item|             ea|        AAUDLOUD|                NULL|
|AAUDLOUDZ00003|MICROPHONE, dynam...|Item|             ea|        AAUDLOUD|IFRC#5010I - ADMI...|
|AAUDLOUDZ00007|AMPLIFIED SPEAKER...|Item|             ea|        AAUDLOUD|IFRC#5010I - ADMI...|
|AAUDLOUDZ00008|DJI MIC 2 (2 TX +...|Item|             ea|        AAUDLOUD|IFRC#5010I - ADMI...|
|AAUDMISCZ00001|Audio Centre (FM,...|Item|             ea|        AAUDMISC|IFRC#5010I - ADMI...|
+--------------+--------------------+----+---------------+----------------+--------------------+
only showing top 5 rows

